# 07 — Đánh giá CNN đã khóa trên test

Chỉ cấu hình và trọng số đã khóa ở06 được đánh giá trên toàn bộ test.
Nhãn chỉ số được đổi về VND trước khi tính metric. Không chọn checkpoint,
ngưỡng softmax hoặc cấu hình mới từ kết quả test.


In [ ]:
# KAGGLE_INPUT_SETUP: tìm code và DATA bằng các mức thư mục giới hạn.
from pathlib import Path
import os, sys

kaggle_input = Path('/kaggle/input')
mounts = []
if kaggle_input.is_dir():
    mounts = [p for pattern in ('datasets/*/*', '*')
              for p in kaggle_input.glob(pattern) if p.is_dir()]
project_hint = os.environ.get('CS231_PROJECT_ROOT')
project_candidates = ([Path(project_hint)] if project_hint else
                      [Path.cwd(), *Path.cwd().parents, *mounts,
                       *(child for mount in mounts for child in mount.glob('*') if child.is_dir())])
project_matches = sorted({p.resolve() for p in project_candidates
                          if (p / 'src/CNN/pipeline.py').is_file()})
if len(project_matches) != 1:
    raise FileNotFoundError('Gắn đúng một Dataset code có src/CNN/pipeline.py; '
                            f'có thể đặt CS231_PROJECT_ROOT. Tìm thấy: {project_matches}')
PROJECT_ROOT = project_matches[0]
sys.path.insert(0, str(PROJECT_ROOT))

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = ([Path(data_hint)] if data_hint else
                   [PROJECT_ROOT / 'DATA', *(p / 'DATA' for p in mounts), *mounts])
data_matches = sorted({p.resolve() for p in data_candidates
                       if (p / 'train/10000').is_dir() and (p / 'dev/10000').is_dir()})
if len(data_matches) != 1:
    raise FileNotFoundError('Gắn đúng Dataset DATA có train/dev; '
                            f'có thể đặt CS231_DATA_ROOT. Tìm thấy: {data_matches}')
DATA_ROOT = data_matches[0]
result_hint = os.environ.get('CS231_CNN_RESULTS_ROOT')
RESULTS_ROOT = (Path(result_hint) if result_hint else
                Path('/kaggle/working/CS231_CNN/results_224') if Path('/kaggle/working').is_dir()
                else PROJECT_ROOT / 'src/CNN/results_224').resolve()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['CS231_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['CS231_DATA_ROOT'] = str(DATA_ROOT)
print('Code:', PROJECT_ROOT)
print('DATA:', DATA_ROOT)
print('Kết quả:', RESULTS_ROOT)


In [ ]:
from src.CNN.runtime import restore_previous_output
restore_previous_output(RESULTS_ROOT, '06', 'selection/locked_cnn_config.json')


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Image, Markdown
from src.CNN.data import CLASSES, assert_protocol
from src.CNN.runtime import mark_stage
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.is_available())


## 1. Kiểm tra khóa và mapping

Kiểm tra hash các artifact đã khóa trước khi đọc test. File khóa xác định cấu hình,
epoch, đường dẫn trọng số và thứ tự sáu mệnh giá. Mô hình được nạp ở chế độ eval,
với BatchNorm dùng thống kê chạy và dropout tắt.


In [ ]:
from src.CNN.pipeline import verify_lock, evaluate_test
lock = verify_lock(RESULTS_ROOT, PROJECT_ROOT)
print('Checkpoint:', lock['checkpoint'], '| epoch:', lock['best_epoch'])
display(pd.DataFrame({'class_index': range(6), 'denomination_vnd': lock['classes_vnd']}))


## 2. Đánh giá toàn bộ test

Ảnh test chỉ resize RGB224, đổi xám nếu cấu hình được chọn yêu cầu và chia255.
Đánh giá lấy argmax logits; không bỏ ảnh khó. Metric được tính một lần trên toàn bộ
dự đoán, kể cả ảnh đúng góp0 vào MAE. Chạy lại khi đã có output hợp lệ sẽ đọc kết quả đã lưu.


In [ ]:
scores, predictions = evaluate_test(DATA_ROOT, RESULTS_ROOT, PROJECT_ROOT)
display(pd.DataFrame([{'split': 'dev', **lock['dev_metrics']},
                      {'split': 'test', **{k: scores[k] for k in ('accuracy', 'macro_f1', 'mae_vnd')}}]))
print('Đúng:', int((predictions.y_true_vnd == predictions.y_pred_vnd).sum()), '/', len(predictions))
print('Tổng chênh lệch VND:', scores['total_gap_vnd'])


## 3. Kết quả theo lớp và cặp nhầm

Ma trận có hàng là nhãn thật, cột là dự đoán, theo mệnh giá tăng dần.
Đọc Precision/Recall/F1 từng lớp cùng bảng cặp nhầm; tổng chênh lệch chia tổng số ảnh
phải bằng MAE. MAE mô tả sai lệch danh nghĩa, chưa phải thiệt hại giao dịch.


In [ ]:
display(pd.DataFrame(scores['per_class']))
display(Image(filename=str(RESULTS_ROOT / 'test_final/confusion_matrix_test.png')))
display(pd.DataFrame(scores['error_pairs']).sort_values('total_gap_vnd', ascending=False)
        if scores['error_pairs'] else pd.DataFrame())


## 4. Giữ hồ sơ lần đánh giá

Output gồm dự đoán từng ảnh, metric, manifest test, khóa đã dùng và thông tin môi trường.
Thời gian ghi là suy luận kèm đọc ảnh của lượt này, không có thời gian fit lại.
Softmax trong CSV chưa hiệu chỉnh thành xác suất đúng.


In [ ]:
display(pd.DataFrame([json.loads((RESULTS_ROOT / 'test_final/run_info.json').read_text())]))
mark_stage(RESULTS_ROOT, '07')
